In [ ]:
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence


# ============================================================
# GRU + VAE FOR STRUCTURAL SVARA REPRESENTATION
# ============================================================
#
# Input per segment:
#   [onehot_CP, onehot_SIL, onehot_STA, duration_rel, cents]
#
# Shape conventions:
#   x:        [batch, seq_len, input_dim]
#   lengths:  [batch]
#
# Notes:
# - This implementation uses a GRU encoder + variational bottleneck + GRU decoder.
# - Decoder supports teacher forcing during training.
# - Loss is masked so padded positions do not contribute.
# - Decoder is conditioned on z at every step by default.
# - Type is assumed to occupy the first 3 dimensions as one-hot ground truth.
# - The code is intentionally simple and readable.


@dataclass
class ModelConfig:
    input_dim = 5
    type_dim = 3
    hidden_dim = 128
    latent_dim = 16
    num_layers = 1
    dropout = 0.0
    max_seq_len = 32
    condition_z_every_step = True
    teacher_forcing_ratio = 1.0

    lambda_type = 1.0
    lambda_dur = 1.0
    lambda_cents = 1.0
    lambda_sum = 0.0

    beta = 1.0
    use_huber_for_continuous = True


# ------------------------------------------------------------
# UTILITIES
# ------------------------------------------------------------


def lengths_to_mask(lengths, max_len=None, device=None):
    if max_len is None:
        max_len = int(lengths.max().item())
    if device is None:
        device = lengths.device
    positions = torch.arange(max_len, device=device).unsqueeze(0)
    mask = positions < lengths.unsqueeze(1)
    return mask


def onehot_to_index(type_onehot):
    return type_onehot.argmax(dim=-1)


def build_start_token(batch_size, input_dim, device):
    return torch.zeros(batch_size, input_dim, device=device)


# ------------------------------------------------------------
# ENCODER
# ------------------------------------------------------------


class SvaraEncoder(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        gru_dropout = cfg.dropout if cfg.num_layers > 1 else 0.0
        self.gru = nn.GRU(
            input_size=cfg.input_dim,
            hidden_size=cfg.hidden_dim,
            num_layers=cfg.num_layers,
            batch_first=True,
            dropout=gru_dropout,
        )
        self.to_mu = nn.Linear(cfg.hidden_dim, cfg.latent_dim)
        self.to_logvar = nn.Linear(cfg.hidden_dim, cfg.latent_dim)

    def forward(self, x, lengths):
        packed = pack_padded_sequence(
            x,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False,
        )
        _, h_n = self.gru(packed)
        h_last = h_n[-1]
        mu = self.to_mu(h_last)
        logvar = self.to_logvar(h_last)
        return mu, logvar


# ------------------------------------------------------------
# DECODER
# ------------------------------------------------------------


class SvaraDecoder(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        decoder_input_dim = cfg.input_dim + (cfg.latent_dim if cfg.condition_z_every_step else 0)

        gru_dropout = cfg.dropout if cfg.num_layers > 1 else 0.0
        self.gru = nn.GRU(
            input_size=decoder_input_dim,
            hidden_size=cfg.hidden_dim,
            num_layers=cfg.num_layers,
            batch_first=True,
            dropout=gru_dropout,
        )

        self.init_hidden = nn.Linear(cfg.latent_dim, cfg.hidden_dim * cfg.num_layers)

        self.type_head = nn.Linear(cfg.hidden_dim, cfg.type_dim)
        self.dur_head = nn.Linear(cfg.hidden_dim, 1)
        self.cents_head = nn.Linear(cfg.hidden_dim, 1)

    def latent_to_hidden(self, z):
        batch_size = z.size(0)
        hidden = self.init_hidden(z)
        hidden = hidden.view(batch_size, self.cfg.num_layers, self.cfg.hidden_dim)
        hidden = hidden.transpose(0, 1).contiguous()
        return hidden

    def step(self, prev_input, hidden, z):
        if self.cfg.condition_z_every_step:
            step_input = torch.cat([prev_input, z], dim=-1)
        else:
            step_input = prev_input

        output, hidden = self.gru(step_input.unsqueeze(1), hidden)
        output = output.squeeze(1)

        type_logits = self.type_head(output)
        dur_pred = self.dur_head(output).squeeze(-1)
        cents_pred = self.cents_head(output).squeeze(-1)

        return type_logits, dur_pred, cents_pred, hidden

    def build_next_input_from_predictions(self, type_logits, dur_pred, cents_pred):
        pred_type_idx = type_logits.argmax(dim=-1)
        pred_type_onehot = F.one_hot(pred_type_idx, num_classes=self.cfg.type_dim).float()
        next_input = torch.cat(
            [pred_type_onehot, dur_pred.unsqueeze(-1), cents_pred.unsqueeze(-1)],
            dim=-1,
        )
        return next_input

    def forward(self, z, target_seq=None, teacher_forcing_ratio=1.0, max_len=None):
        batch_size = z.size(0)
        device = z.device
        hidden = self.latent_to_hidden(z)

        if max_len is None:
            if target_seq is None:
                max_len = self.cfg.max_seq_len
            else:
                max_len = target_seq.size(1)

        prev_input = build_start_token(batch_size, self.cfg.input_dim, device)

        type_logits_all = []
        dur_all = []
        cents_all = []

        for t in range(max_len):
            type_logits, dur_pred, cents_pred, hidden = self.step(prev_input, hidden, z)

            type_logits_all.append(type_logits)
            dur_all.append(dur_pred)
            cents_all.append(cents_pred)

            if self.training and target_seq is not None:
                use_teacher = torch.rand(1, device=device).item() < teacher_forcing_ratio
            else:
                use_teacher = False

            if use_teacher:
                prev_input = target_seq[:, t, :]
            else:
                prev_input = self.build_next_input_from_predictions(type_logits, dur_pred, cents_pred)

        type_logits_all = torch.stack(type_logits_all, dim=1)
        dur_all = torch.stack(dur_all, dim=1)
        cents_all = torch.stack(cents_all, dim=1)

        return {
            "type_logits": type_logits_all,
            "duration": dur_all,
            "cents": cents_all,
        }


# ------------------------------------------------------------
# FULL MODEL
# ------------------------------------------------------------


class SvaraGRUVAE(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.encoder = SvaraEncoder(cfg)
        self.decoder = SvaraDecoder(cfg)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        z = mu + std * eps
        return z

    def forward(self, x, lengths, teacher_forcing_ratio=None):
        if teacher_forcing_ratio is None:
            teacher_forcing_ratio = self.cfg.teacher_forcing_ratio

        mu, logvar = self.encoder(x, lengths)
        z = self.reparameterize(mu, logvar)

        decoded = self.decoder(
            z,
            target_seq=x,
            teacher_forcing_ratio=teacher_forcing_ratio,
            max_len=x.size(1),
        )

        return {
            "mu": mu,
            "logvar": logvar,
            "z": z,
            "type_logits": decoded["type_logits"],
            "duration": decoded["duration"],
            "cents": decoded["cents"],
        }

    def generate(self, batch_size=1, z=None, max_len=None, device=None):
        if device is None:
            device = next(self.parameters()).device
        if z is None:
            z = torch.randn(batch_size, self.cfg.latent_dim, device=device)
        if max_len is None:
            max_len = self.cfg.max_seq_len

        self.eval()
        with torch.no_grad():
            decoded = self.decoder(
                z,
                target_seq=None,
                teacher_forcing_ratio=0.0,
                max_len=max_len,
            )
            pred_type_idx = decoded["type_logits"].argmax(dim=-1)
            pred_type_onehot = F.one_hot(pred_type_idx, num_classes=self.cfg.type_dim).float()
            generated = torch.cat(
                [pred_type_onehot, decoded["duration"].unsqueeze(-1), decoded["cents"].unsqueeze(-1)],
                dim=-1,
            )
        return {
            "z": z,
            "generated": generated,
            "type_logits": decoded["type_logits"],
            "duration": decoded["duration"],
            "cents": decoded["cents"],
        }


# ------------------------------------------------------------
# LOSSES
# ------------------------------------------------------------


def reconstruction_loss(outputs, targets, lengths, cfg):
    device = targets.device
    batch_size, seq_len, _ = targets.shape
    mask = lengths_to_mask(lengths, max_len=seq_len, device=device).float()

    target_type_idx = onehot_to_index(targets[:, :, :cfg.type_dim])
    target_duration = targets[:, :, cfg.type_dim]
    target_cents = targets[:, :, cfg.type_dim + 1]

    type_logits = outputs["type_logits"]
    pred_duration = outputs["duration"]
    pred_cents = outputs["cents"]

    type_loss_all = F.cross_entropy(
        type_logits.reshape(-1, cfg.type_dim),
        target_type_idx.reshape(-1),
        reduction="none",
    ).view(batch_size, seq_len)

    if cfg.use_huber_for_continuous:
        dur_loss_all = F.smooth_l1_loss(pred_duration, target_duration, reduction="none")
        cents_loss_all = F.smooth_l1_loss(pred_cents, target_cents, reduction="none")
    else:
        dur_loss_all = F.mse_loss(pred_duration, target_duration, reduction="none")
        cents_loss_all = F.mse_loss(pred_cents, target_cents, reduction="none")

    type_loss = (type_loss_all * mask).sum() / mask.sum().clamp_min(1.0)
    dur_loss = (dur_loss_all * mask).sum() / mask.sum().clamp_min(1.0)
    cents_loss = (cents_loss_all * mask).sum() / mask.sum().clamp_min(1.0)

    loss = (
        cfg.lambda_type * type_loss
        + cfg.lambda_dur * dur_loss
        + cfg.lambda_cents * cents_loss
    )

    if cfg.lambda_sum > 0.0:
        pred_duration_sum = (pred_duration * mask).sum(dim=1)
        sum_loss = ((pred_duration_sum - 1.0) ** 2).mean()
        loss = loss + cfg.lambda_sum * sum_loss
    else:
        sum_loss = torch.tensor(0.0, device=device)

    return loss, {
        "type_loss": type_loss.detach(),
        "dur_loss": dur_loss.detach(),
        "cents_loss": cents_loss.detach(),
        "sum_loss": sum_loss.detach(),
    }


def kl_loss(mu, logvar):
    kl_per_sample = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1)
    return kl_per_sample.mean()


def total_vae_loss(outputs, targets, lengths, cfg, beta=None):
    if beta is None:
        beta = cfg.beta

    recon, recon_parts = reconstruction_loss(outputs, targets, lengths, cfg)
    kl = kl_loss(outputs["mu"], outputs["logvar"])
    total = recon + beta * kl

    stats = {
        "total_loss": total.detach(),
        "recon_loss": recon.detach(),
        "kl_loss": kl.detach(),
    }
    stats.update(recon_parts)
    return total, stats


# ------------------------------------------------------------
# KL ANNEALING
# ------------------------------------------------------------


def linear_kl_beta(step, warmup_steps):
    if warmup_steps <= 0:
        return 1.0
    return min(1.0, step / float(warmup_steps))


# ------------------------------------------------------------
# EXAMPLE TRAIN STEP
# ------------------------------------------------------------


def train_step(model, batch_x, batch_lengths, optimizer, global_step, warmup_steps=1000):
    model.train()
    optimizer.zero_grad()

    beta = linear_kl_beta(global_step, warmup_steps)
    outputs = model(batch_x, batch_lengths)
    loss, stats = total_vae_loss(outputs, batch_x, batch_lengths, model.cfg, beta=beta)

    loss.backward()
    optimizer.step()

    stats["beta"] = torch.tensor(beta, device=batch_x.device)
    return stats


# ------------------------------------------------------------
# MINIMAL USAGE EXAMPLE
# ------------------------------------------------------------


if __name__ == "__main__":
    cfg = ModelConfig()
    model = SvaraGRUVAE(cfg)

    batch_size = 4
    seq_len = 10

    x = torch.randn(batch_size, seq_len, cfg.input_dim)
    x[:, :, :cfg.type_dim] = 0.0

    random_types = torch.randint(0, cfg.type_dim, (batch_size, seq_len))
    x[:, :, :cfg.type_dim] = F.one_hot(random_types, num_classes=cfg.type_dim).float()

    lengths = torch.tensor([10, 8, 6, 9])

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    stats = train_step(
        model=model,
        batch_x=x,
        batch_lengths=lengths,
        optimizer=optimizer,
        global_step=100,
        warmup_steps=1000,
    )

    print("Train stats:")
    for key, value in stats.items():
        print(key, float(value))

    generated = model.generate(batch_size=2)
    print("Generated shape:", generated["generated"].shape)
